In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import numpy as np

# --- 1. CONFIGURATION ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

# KEY CHANGES FOR STABILITY:
P = 97                 # Prime number (Classic Grokking setup)
BATCH_SIZE = 512       # Batching helps noise
LR = 1e-3
WEIGHT_DECAY = 1.0     # The "Magic" Number for this specific task
EPOCHS = 50000
HIDDEN_DIM = 128
NUM_LAYERS = 2
REPORT_EVERY = 200

# --- 2. DATA: Modulo 97 Addition ---
# Dataset size jumps from 100 -> ~9,400.
# This makes memorization "expensive" enough to force grokking.
data = []
labels = []
for i in range(P):
    for j in range(P):
        data.append([i, j])
        labels.append((i + j) % P) # Modulo addition

X = torch.tensor(data).to(device)
Y = torch.tensor(labels).to(device)

# 50/50 Split
perm = torch.randperm(len(X))
split = int(len(X) * 0.1)
X_train, Y_train = X[perm[:split]], Y[perm[:split]]
X_test, Y_test = X[perm[split:]], Y[perm[split:]]

# --- 3. MODEL ---
class GrokTransformer(nn.Module):
    def _init_(self):
        super()._init_()
        self.emb = nn.Embedding(P, HIDDEN_DIM)
        self.pos = nn.Parameter(torch.randn(1, 2, HIDDEN_DIM))
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(HIDDEN_DIM, 4, HIDDEN_DIM*4, dropout=0.0, batch_first=True),
            num_layers=NUM_LAYERS
        )
        self.head = nn.Linear(HIDDEN_DIM, P)
        # Note: We output P classes (0-96)

    def forward(self, x):
        x = self.emb(x) + self.pos
        x = self.encoder(x)
        return self.head(x.mean(dim=1))

model = GrokTransformer().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()

# --- 4. LIVE GRAPH LOOP ---
train_accs = []
test_accs = []
epochs = []
loss_history = []

print("Starting Modulo-97 Experiment...")

for epoch in range(EPOCHS + 1):
    model.train()

    # Batch training (Better for this data size)
    permutation = torch.randperm(X_train.size(0))
    for i in range(0, X_train.size(0), BATCH_SIZE):
        indices = permutation[i:i+BATCH_SIZE]
        batch_x, batch_y = X_train[indices], Y_train[indices]

        optimizer.zero_grad()
        out = model(batch_x)
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()

    # --- VISUALIZATION ---
    if epoch % REPORT_EVERY == 0:
        model.eval()
        with torch.no_grad():
            # Evaluation on full sets
            # We do this in chunks to avoid GPU OOM if memory is tight,
            # but for 97^2 it fits in memory easily.
            tr_acc = (model(X_train).argmax(1) == Y_train).float().mean().item() * 100
            te_acc = (model(X_test).argmax(1) == Y_test).float().mean().item() * 100

            train_accs.append(tr_acc)
            test_accs.append(te_acc)
            epochs.append(epoch)

            clear_output(wait=True)
            plt.figure(figsize=(10, 6))

            # Plot
            plt.plot(epochs, train_accs, label='Train (Memorization)', color='cyan', alpha=0.8)
            plt.plot(epochs, test_accs, label='Test (Grokking)', color='orange', linewidth=2.5)

            plt.ylim(-5, 105)
            plt.title(f"Modulo {P} Grokking | Epoch {epoch}", fontsize=16)
            plt.xlabel("Epochs")
            plt.ylabel("Accuracy %")
            plt.legend(loc='lower right')
            plt.grid(True, alpha=0.3)

            # Status Text
            status = "Status: Learning..."
            if tr_acc > 95 and te_acc < 60:
                status = "Status: MEMORIZED. Waiting for Grok..."
            elif te_acc > 90:
                status = "Status: GROKKED!"

            plt.text(0, 110, status, fontsize=12, fontweight='bold')
            plt.show()

            if te_acc > 99.999999:
                print(f"Grokking Complete at Epoch {epoch}!")
                break

In [7]:
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)


NVIDIA GeForce RTX 5050 Laptop GPU
12.1


In [1]:
print("f")

f


In [2]:

!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
